In [ ]:
import os
# Prevent memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import math
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import vgg19, VGG19_Weights
from torchvision.utils import save_image
from PIL import Image
import random
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# ==========================================
# 1. SETUP & HYPERPARAMETERS
# ==========================================
NUM_GPUS = torch.cuda.device_count()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Initializing Stabilized Training on {NUM_GPUS} GPUs")

BATCH_SIZE = 4 * max(1, NUM_GPUS) 
NUM_EPOCHS = 20 
WARMUP_EPOCHS = 3
LR = 2e-4
HR_SIZE = 256
LR_SIZE = 64 

IMAGE_DIR = "/kaggle/input/datasets/rahulbhalley/ffhq-1024x1024/images1024x1024" 
WORKING_DIR = "/kaggle/working/"

# ==========================================
# 2. FAST CPU DATASET LOADER
# ==========================================
class RealWorldDegradationDataset(Dataset):
    def __init__(self, root_dir):
        self.image_files = [os.path.join(root_dir, f) for f in os.listdir(root_dir) if f.lower().endswith(('png', 'jpg', 'jpeg'))]
        if not self.image_files:
            raise RuntimeError(f"No images found in {root_dir}! Check IMAGE_DIR.")
            
        self.hr_transform = transforms.Compose([
            transforms.RandomCrop(HR_SIZE, pad_if_needed=True),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        clean_img = Image.open(self.image_files[idx]).convert("RGB")
        return self.hr_transform(clean_img)

# ==========================================
# 3. RRDBNET GENERATOR (8-Block Variant)
# ==========================================
class DenseResidualBlock(nn.Module):
    def __init__(self, filters=64, res_scale=0.2):
        super().__init__()
        self.res_scale = res_scale
        self.conv1 = nn.Conv2d(filters, 32, 3, 1, 1)
        self.conv2 = nn.Conv2d(filters + 32, 32, 3, 1, 1)
        self.conv3 = nn.Conv2d(filters + 64, 32, 3, 1, 1)
        self.conv4 = nn.Conv2d(filters + 96, 32, 3, 1, 1)
        self.conv5 = nn.Conv2d(filters + 128, filters, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        c1 = self.lrelu(self.conv1(x))
        c2 = self.lrelu(self.conv2(torch.cat((x, c1), 1)))
        c3 = self.lrelu(self.conv3(torch.cat((x, c1, c2), 1)))
        c4 = self.lrelu(self.conv4(torch.cat((x, c1, c2, c3), 1)))
        c5 = self.conv5(torch.cat((x, c1, c2, c3, c4), 1))
        return c5.mul(self.res_scale) + x

class RRDB(nn.Module):
    def __init__(self, filters=64):
        super().__init__()
        self.rdb1 = DenseResidualBlock(filters)
        self.rdb2 = DenseResidualBlock(filters)
        self.rdb3 = DenseResidualBlock(filters)

    def forward(self, x):
        out = self.rdb1(x)
        out = self.rdb2(out)
        out = self.rdb3(out)
        return out.mul(0.2) + x

class RRDBNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, filters=64, num_rrdb=8):
        super().__init__()
        self.conv_first = nn.Conv2d(in_channels, filters, 3, 1, 1)
        self.rrdb_blocks = nn.Sequential(*[RRDB(filters) for _ in range(num_rrdb)])
        self.conv_trunk = nn.Conv2d(filters, filters, 3, 1, 1)
        
        self.upsample1 = nn.Sequential(nn.Conv2d(filters, filters * 4, 3, 1, 1), nn.PixelShuffle(2), nn.LeakyReLU(0.2, inplace=True))
        self.upsample2 = nn.Sequential(nn.Conv2d(filters, filters * 4, 3, 1, 1), nn.PixelShuffle(2), nn.LeakyReLU(0.2, inplace=True))
        
        self.conv_last = nn.Sequential(
            nn.Conv2d(filters, filters, 3, 1, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(filters, out_channels, 3, 1, 1)
        )

    def forward(self, x):
        feat = self.conv_first(x)
        trunk = self.conv_trunk(self.rrdb_blocks(feat))
        feat = feat + trunk
        feat = self.upsample1(feat) 
        feat = self.upsample2(feat) 
        return torch.clamp(self.conv_last(feat), 0, 1)

# ==========================================
# 4. PERCEPTUAL LOSS & DISCRIMINATOR
# ==========================================
class VGGLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = vgg19(weights=VGG19_Weights.DEFAULT).features[:36].eval().to(DEVICE)
        for param in vgg.parameters(): param.requires_grad = False
        self.vgg = vgg
        self.criterion = nn.MSELoss()
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, gen_hr, real_hr):
        gen_features = self.vgg((gen_hr - self.mean) / self.std)
        real_features = self.vgg((real_hr - self.mean) / self.std)
        return self.criterion(gen_features, real_features)

class Discriminator(nn.Module):
    def __init__(self, in_channels=3):
        super().__init__()
        def block(in_filters, out_filters, stride=1, normalize=True):
            layers = [nn.Conv2d(in_filters, out_filters, 3, stride, 1, bias=False)]
            if normalize: layers.append(nn.BatchNorm2d(out_filters))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        self.model = nn.Sequential(
            *block(in_channels, 64, normalize=False), *block(64, 64, stride=2),
            *block(64, 128), *block(128, 128, stride=2),
            *block(128, 256), *block(256, 256, stride=2),
            *block(256, 512), *block(512, 512, stride=2),
            nn.Conv2d(512, 1, 3, 1, 1)
        )

    def forward(self, img):
        return self.model(img)

# ==========================================
# 5. METRICS & VISUALIZATION
# ==========================================
def calculate_psnr(img1, img2):
    mse = torch.mean((img1 - img2) ** 2)
    if mse == 0: return 100.0
    return 20 * math.log10(1.0) - 10 * math.log10(mse.item())

def plot_and_save_logs(history, folder):
    df = pd.DataFrame(history)
    df.to_csv(f"{folder}training_logs.csv", index=False)
    
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(df['epoch'], df['g_loss'], label='Generator Loss', color='blue')
    plt.plot(df['epoch'], df['d_loss'], label='Discriminator Loss', color='red')
    plt.title('GAN Loss Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(df['epoch'], df['psnr'], label='PSNR (dB)', color='green', linewidth=2)
    plt.title('Image Quality (PSNR)')
    plt.xlabel('Epoch')
    plt.ylabel('PSNR (Decibels)')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(f"{folder}training_graphs.png")
    plt.close() 

def save_examples(gen, lr, hr, epoch):
    gen.eval()
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            sr = gen(lr)
        lr_up = F.interpolate(lr, size=(HR_SIZE, HR_SIZE), mode='bicubic')
        stacked = torch.cat([lr_up[:4], sr[:4], hr[:4]], dim=3)
        save_image(stacked, f"{WORKING_DIR}eval_epoch_{epoch}.png")
    gen.train()

# ==========================================
# 6. STABILIZED TRAINING LOOP (WITH WARMUP)
# ==========================================
def train():
    dataset = RealWorldDegradationDataset(IMAGE_DIR)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)

    gen = RRDBNet().to(DEVICE)
    disc = Discriminator().to(DEVICE)

    if NUM_GPUS > 1:
        gen = nn.DataParallel(gen)
        disc = nn.DataParallel(disc)

    opt_gen = optim.Adam(gen.parameters(), lr=LR, betas=(0.9, 0.999))
    opt_disc = optim.Adam(disc.parameters(), lr=LR, betas=(0.9, 0.999))

    l1_loss = nn.L1Loss()
    bce_loss = nn.BCEWithLogitsLoss()
    vgg_loss = VGGLoss().to(DEVICE)
    scaler = torch.amp.GradScaler('cuda')

    history = []

    for epoch in range(1, NUM_EPOCHS + 1):
        loop = tqdm(loader, leave=True)
        epoch_d_loss, epoch_g_loss, epoch_psnr = 0, 0, 0

        for idx, hr in enumerate(loop):
            hr = hr.to(DEVICE)

            # Fast GPU Degradation
            with torch.no_grad():
                lr = F.interpolate(hr, size=(LR_SIZE, LR_SIZE), mode='bicubic', align_corners=False)
                noise = torch.randn_like(lr) * random.uniform(0.01, 0.05)
                lr = torch.clamp(lr + noise, 0, 1)

            # --------------------------------------------------
            # PHASE 1: GENERATOR WARMUP (PREVENTS BLACK IMAGES)
            # --------------------------------------------------
            if epoch <= WARMUP_EPOCHS:
                opt_gen.zero_grad()
                with torch.amp.autocast('cuda'):
                    sr = gen(lr)
                    g_loss = l1_loss(sr, hr) # Pure pixel math, no GAN instability

                scaler.scale(g_loss).backward()
                
                # Gradient Clipping to prevent explosion
                scaler.unscale_(opt_gen)
                torch.nn.utils.clip_grad_norm_(gen.parameters(), max_norm=1.0)
                
                scaler.step(opt_gen)
                scaler.update()
                
                d_loss = torch.tensor(0.0)

            # --------------------------------------------------
            # PHASE 2: FULL GAN TRAINING
            # --------------------------------------------------
            else:
                # Train Discriminator
                opt_disc.zero_grad()
                with torch.amp.autocast('cuda'):
                    sr = gen(lr)
                    real_preds = disc(hr)
                    fake_preds = disc(sr.detach())
                    
                    d_real_loss = bce_loss(real_preds, torch.ones_like(real_preds) - random.uniform(0, 0.1)) 
                    d_fake_loss = bce_loss(fake_preds, torch.zeros_like(fake_preds))
                    d_loss = d_real_loss + d_fake_loss

                scaler.scale(d_loss).backward()
                scaler.step(opt_disc)

                # Train Generator
                opt_gen.zero_grad()
                with torch.amp.autocast('cuda'):
                    fake_preds = disc(sr)
                    
                    l_adv = bce_loss(fake_preds, torch.ones_like(fake_preds))
                    l_vgg = vgg_loss(sr, hr)
                    l_pixel = l1_loss(sr, hr)
                    
                    g_loss = (l_pixel * 1.0) + (l_vgg * 1.0) + (l_adv * 0.005)

                scaler.scale(g_loss).backward()
                
                # Gradient Clipping 
                scaler.unscale_(opt_gen)
                torch.nn.utils.clip_grad_norm_(gen.parameters(), max_norm=1.0)
                
                scaler.step(opt_gen)
                scaler.update()

            # Tracking Metrics
            current_psnr = calculate_psnr(sr.detach(), hr)
            epoch_d_loss += d_loss.item()
            epoch_g_loss += g_loss.item()
            epoch_psnr += current_psnr
            
            loop.set_postfix(Epoch=epoch, D_loss=d_loss.item(), G_loss=g_loss.item(), PSNR=f"{current_psnr:.2f}dB")

        # Save continuous logs
        history.append({
            "epoch": epoch, 
            "d_loss": epoch_d_loss / len(loader), 
            "g_loss": epoch_g_loss / len(loader), 
            "psnr": epoch_psnr / len(loader)
        })
        
        df = pd.DataFrame(history)
        df.to_csv(f"{WORKING_DIR}training_logs.csv", index=False)

        # --------------------------------------------------
        # TRIGGER SAVES EVERY 5 EPOCHS
        # --------------------------------------------------
        if epoch % 5 == 0 or epoch == NUM_EPOCHS:
            plot_and_save_logs(history, WORKING_DIR)
            save_examples(gen, lr, hr, epoch)
            state_dict = gen.module.state_dict() if NUM_GPUS > 1 else gen.state_dict()
            torch.save(state_dict, f"{WORKING_DIR}RRDB_4x_epoch_{epoch}.pth")
            print(f"\nSaved Epoch {epoch}: Model, Plots, and Images exported.")

        torch.cuda.empty_cache()

    print("\nTraining Complete!")

if __name__ == "__main__":
    train()